In [1]:


# 8. RANDOM FOREST EN RÉGRESSION : les ventes de glaces
#    Critère = SSE = Σ (valeur − moyenne du groupe)²
#    Feuille = moyenne du groupe ; forêt = moyenne des arbres

VENTES = {12: 40, 18: 60, 26: 200, 32: 240}     # température (°C) -> ventes (TND)
ECH_GLACES = [
    [12, 18, 32, 32],    # arbre 1
    [12, 12, 26, 32],    # arbre 2
    [18, 18, 26, 26],    # arbre 3
]

def moyenne(v):
    return sum(v) / len(v)

def sse(v):
    m = moyenne(v)
    return sum((x - m) ** 2 for x in v)

def apprendre_arbre_reg(echantillon):
    temps = sorted(set(echantillon))
    meilleur = None
    for a, b in zip(temps, temps[1:]):
        seuil = (a + b) / 2
        gauche = [VENTES[t] for t in echantillon if t <= seuil]
        droite = [VENTES[t] for t in echantillon if t > seuil]
        erreur = sse(gauche) + sse(droite)
        print(f"    T <= {seuil:g} : gauche {gauche} droite {droite} -> SSE = {erreur:g}")
        if meilleur is None or erreur < meilleur["sse"]:
            meilleur = {"seuil": seuil, "sse": erreur,
                        "gauche": moyenne(gauche), "droite": moyenne(droite)}
    print(f"  -> seuil gardé : T <= {meilleur['seuil']:g} ; "
          f"feuilles {meilleur['gauche']:g} / {meilleur['droite']:g}")
    return meilleur

print("\n RÉGRESSION : VENTES DE GLACES ")
foret_reg = []
for i, ech in enumerate(ECH_GLACES, start=1):
    print(f"\n  Arbre {i} — échantillon {ech}")
    foret_reg.append(apprendre_arbre_reg(ech))

T = 28
preds = [a["gauche"] if T <= a["seuil"] else a["droite"] for a in foret_reg]
print(f"\n  Pour {T} °C : arbres -> {[f'{p:g}' for p in preds]}")
print(f"  Forêt : ({' + '.join(f'{p:g}' for p in preds)}) / {len(preds)} = {moyenne(preds):g} TND")




=== RÉGRESSION : VENTES DE GLACES ===

  Arbre 1 — échantillon [12, 18, 32, 32]
    T <= 15 : gauche [40] droite [60, 240, 240] -> SSE = 21600
    T <= 25 : gauche [40, 60] droite [240, 240] -> SSE = 200
  -> seuil gardé : T <= 25 ; feuilles 50 / 240

  Arbre 2 — échantillon [12, 12, 26, 32]
    T <= 19 : gauche [40, 40] droite [200, 240] -> SSE = 800
    T <= 29 : gauche [40, 40, 200] droite [240] -> SSE = 17066.7
  -> seuil gardé : T <= 19 ; feuilles 40 / 220

  Arbre 3 — échantillon [18, 18, 26, 26]
    T <= 22 : gauche [60, 60] droite [200, 200] -> SSE = 0
  -> seuil gardé : T <= 22 ; feuilles 60 / 200

  Pour 28 °C : arbres -> ['240', '220', '200']
  Forêt : (240 + 220 + 200) / 3 = 220 TND
